# Dense, Sparce, Hybrid and Reranking
- Implementar busqueda hibrida con filtros
- Aplicar reranking para mejores resultados

In [1]:
import sys
sys.path.append("D:/Cursos/Agentes/financial_deep_research_agent")

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore, RetrievalMode, FastEmbedSparse
from langchain.messages import HumanMessage, SystemMessage

#metadata extraction from llm
from agent.schema import ChunkMetadata

d:\Cursos\Agentes\financial_deep_research_agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#configuration
COLLECTION_NAME = "financial_docs"
EMBEDDING_MODEL = "models/gemini-embedding-001"
LLM_MODEL = "gemini-2.5-flash"
RERANKER_MODEL = "BAAI/bge-reranker-base"
url = os.getenv("QDRANT_URL")

In [4]:
#initialize llm
llm = ChatGoogleGenerativeAI(model=LLM_MODEL)
#gemini embedding
embedding = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)
#sparse embedding
sparse_embedding = FastEmbedSparse(model_name="Qdrant/bm25")

#connect to existing collection
vector_store = QdrantVectorStore.from_existing_collection(
    collection_name=COLLECTION_NAME,
    embedding=embedding,
    sparse_embedding=sparse_embedding,
    url=url,
    retrieval_mode=RetrievalMode.HYBRID
)
vector_store

### Filter Extraction with LLM

In [5]:
def extract_filters(user_query: str):
    system_prompt = f"""
        Extract metadata filters from the query. Return None for fields not mentioned.

            #### EXAMPLES
            COMPANY MAPPINGS:
            - Amazon/AMZN -> amazon
            - Google/Alphabet/GOOGL/GOOG -> google
            - Apple/AAPL -> apple
            - Microsoft/MSFT -> microsoft
            - Tesla/TSLA -> tesla
            - Nvidia/NVDA -> nvidia
            - Meta/Facebook/FB -> meta

            DOC TYPE:
            - Annual report -> 10-k
            - Quarterly report -> 10-q
            - Current report -> 8-k

            EXAMPLES:
            "Amazon Q3 2024 revenue" -> {{"company_name": "amazon", "doc_type": "10-q", "fiscal_year": 2024, "fiscal_quarter": "q3"}}
            "Apple 2023 annual report" -> {{"company_name": "apple", "doc_type": "10-k", "fiscal_year": 2023}}
            "Tesla profitability" -> {{"company_name": "tesla"}}

            Extract metadata based on the user query only:
        """
    structured_llm = llm.with_structured_output(ChunkMetadata)
    metadata = structured_llm.invoke([SystemMessage(system_prompt), HumanMessage(f"user query: {user_query}")])
    filters = metadata.model_dump(exclude_none=True)
    return filters

In [24]:
# query = "what is amazon's revenue in 2024 in q1"
# filters = extract_filters(query)

In [7]:
filters

{'company_name': 'amazon',
 'fiscal_year': '2024',
 'fical_quarter': <FiscalQuarter.Q1: 'q1'>}

### Retrieval Fucntions

In [8]:
#metadata filtering
from qdrant_client.models import Filter, FieldCondition, MatchValue

# [FieldCondition(key=f"metadata.{key}", match=MatchValue(value=value)) for key, value in filters.items()]

In [25]:
def hybrid_search(query: str, k: int = 10, filters: dict = None):
    """Hybrid search (dense + sparse vectors)"""
    filters = extract_filters(query)
    qdrant_filter = None
    
    if filters:
        condition = [FieldCondition(key=f"metadata.{key}", match=MatchValue(value=value)) for key, value in filters.items()]
        qdrant_filter = Filter(must=condition)
    
    result = vector_store.similarity_search(query=query, k=k, filter=qdrant_filter)
    return result

In [26]:
query = "what is amazon's cashflow in 2024 in q1"

results = hybrid_search(query, k=10)

In [16]:
results

[Document(metadata={'company_name': 'amazon', 'doc_type': '10-q', 'fical_quarter': 'q1', 'fiscal_year': '2024', 'content_type': 'text', 'file_hash': '7ef1ccdf7b4e355db01cc086b7078c4258749bf4d7e5d35d626ccbb6d0108014', 'source_file': 'amazon 10-q q1 2024.md', 'page': 56, '_id': 'd4d30cd0-7dc9-4e5d-b0a4-8c8ee190e026', '_collection_name': 'financial_docs'}, page_content="\n\nThe increase in fulfillment costs in Q1 2024, compared to the comparable prior year period, is primarily due to increased sales and investments in our fulfillment network, partially offset by fulfillment network efficiencies. Changes in foreign exchange rates increased fulfillment costs by $14 million for Q1 2024.\n\nWe seek to expand our fulfillment network to accommodate a greater selection and in-stock inventory levels and to meet anticipated shipment volumes from sales of our own products as well as sales by third parties for which we provide the fulfillment services. We regularly evaluate our facility requirements

### Reranking

In [22]:
#reranking for better result
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

def rerank_results(query: str, documents: list, top_k:int = 5):
    """Rerank documents using cross-encoder
    Return:
        List of (score, Documents) tuples sorted by relevance"""
    
    reranker = HuggingFaceCrossEncoder(
        model_name=RERANKER_MODEL,
        model_kwargs={'device':'cpu'}
    )

    query_doc_pairs = [(query, doc.page_content) for doc in documents]

    scores = reranker.score(query_doc_pairs)
    reranked = sorted(zip(scores, documents), key=lambda x: x[0], reverse=True)
    reranked = reranked[:top_k]
    return [rank[1] for rank in reranked]
    
    

In [23]:
response = rerank_results(query, results)
response

[Document(metadata={'company_name': 'amazon', 'doc_type': '10-q', 'fical_quarter': 'q1', 'fiscal_year': '2024', 'content_type': 'text', 'file_hash': '7ef1ccdf7b4e355db01cc086b7078c4258749bf4d7e5d35d626ccbb6d0108014', 'source_file': 'amazon 10-q q1 2024.md', 'page': 56, '_id': 'd4d30cd0-7dc9-4e5d-b0a4-8c8ee190e026', '_collection_name': 'financial_docs'}, page_content="\n\nThe increase in fulfillment costs in Q1 2024, compared to the comparable prior year period, is primarily due to increased sales and investments in our fulfillment network, partially offset by fulfillment network efficiencies. Changes in foreign exchange rates increased fulfillment costs by $14 million for Q1 2024.\n\nWe seek to expand our fulfillment network to accommodate a greater selection and in-stock inventory levels and to meet anticipated shipment volumes from sales of our own products as well as sales by third parties for which we provide the fulfillment services. We regularly evaluate our facility requirements

### Example: Dynamic Filters


In [31]:
query = "what is the revenue of apple in 2023"

results = hybrid_search(query, k=5)
results

[Document(metadata={'company_name': 'apple', 'doc_type': '10-q', 'fical_quarter': 'q4', 'fiscal_year': '2023', 'content_type': 'text', 'file_hash': '28f96f4d687dccb3765391bba195fc7f70b3f2b6e51f883cfd7a0750429bf88c', 'source_file': 'apple 10-q q4 2023.md', 'page': 11, '_id': 'b2617ca5-f3b3-44cb-8189-0f556f23fb29', '_collection_name': 'financial_docs'}, page_content="\n\n## Apple Inc.\n\n## Notes to Condensed Consolidated Financial Statements (Unaudited)\n\n## Note 1 - Summary of Significant Accounting Policies\n\n## Basis of Presentation and Preparation\n\nThe condensed consolidated financial statements include the accounts of Apple Inc. and its wholly owned subsidiaries (collectively 'Apple' or the 'Company'). In the opinion of the Company's management, the condensed consolidated financial statements reflect all adjustments, which are normal and recurring in nature, necessary for fair financial statement presentation. The preparation of these condensed consolidated financial statements